In [1]:
import wandb
wandb.login()

wandb: Currently logged in as: rediska0123. Use `wandb login --relogin` to force relogin


True

In [2]:
import numpy as np
import pandas as pd
from collections import defaultdict

INDEX_ORDER = [
    ('RandomBaselineClaim', 'Random'),
    ('MaximumClaimProbability', 'MCP'),
    ('PerplexityClaim', 'Perplexity'),
    ('MaxTokenEntropyClaim', 'Max Token Entropy'),
#     ('CCP_claim_no_context', 'CCP'),
    ('CCP_claim_fact_pref', 'CCP'),
    ('LuqClaimEstimator_mean_logits_reverse=False', 'UHead (mean prob., bio)'),
    ('LuqClaimEstimator_mean_reverse=False', 'UHead (mean logits, bio)'),
    ('LuqClaimEstimator_prod_geom_reverse=False', 'UHead (prob. prod., bio)'),
    ('LuqClaimEstimatorDummy_claim', 'UHead'),
]

def get_runs(prefix, model, n=500):
    api = wandb.Api()
    runs = api.runs("nlpresearch.group/luq")
    runs_by_split = {}
    for run in runs[:n]:
        print(run.name)
        if run.name.startswith(f'{prefix}:') and run.name.endswith(model):
            split = run.name.split(':')[-1].split()[0]
            runs_by_split[split] = run.id
    print('Generating results table for runs:', runs_by_split)
    return runs_by_split

def get_df_from_runs(runs_by_split, splits, metric='roc-auc'):
    df = {}
    api = wandb.Api()
    for split in splits:
        df[split] = {}
        if split not in runs_by_split:
            continue
        run_id = runs_by_split[split]
        run = api.run(f"nlpresearch.group/luq/{run_id}")

        history = run.history(samples=None)
        for column in history.columns:
            if not column.startswith("('claim', "):
                continue
            _, method, _, m = map(lambda x: x[1:-1], column[1:-1].split(', '))
            if m != metric:
                continue
            val = [v for v in history[column].tolist() if not np.isnan(v)]
            assert len(val) <= 1
            val = val[0] if len(val) else None
            str_val = str(round(val, 2))
            while len(str_val) < 4:
                str_val += '0'
            df[split][method] = str_val
    
    all_methods = list(set(k for d in df.values() for k in d.keys()))
    print(all_methods)
    index_order = [(k, v) for k, v in INDEX_ORDER if k in all_methods]
    df = pd.DataFrame(df).rename(index={k: v for k, v in index_order}).reindex([v for _, v in index_order])
    
    return df

In [3]:
df = get_df_from_runs(
    {'events': 'vghzvbhy', 'landmarks': 'x02w6etn', 'artworks': '5yjyl1n9', 'books': '53zqnyu0', 'inventions': '3dudlhp9', 'movies': 'dipa5kwz', 'cities': 'm7whxhbm', 'persons': 'wjk3yyn3'},
#     get_runs('test_multidomain_claim', 'llama3', 5000),
    splits=["persons", "cities", "movies", "inventions", "books", "artworks", "landmarks", "events"],
    metric='pr-auc'
)
print(df.to_latex().replace(' NaN ', ' '))
df

['MaximumClaimProbability', 'MaxTokenEntropyClaim', 'PerplexityClaim', 'LuqClaimEstimatorDummy_claim', 'RandomBaselineClaim', 'CCP_claim_fact_pref']
\begin{tabular}{lllllllll}
\toprule
 & persons & cities & movies & inventions & books & artworks & landmarks & events \\
\midrule
Random & 0.22 & 0.20 & 0.08 & 0.17 & 0.09 & 0.18 & 0.12 & 0.10 \\
MCP & 0.35 & 0.33 & 0.14 & 0.27 & 0.15 & 0.30 & 0.21 & 0.12 \\
Perplexity & 0.32 & 0.28 & 0.13 & 0.24 & 0.11 & 0.32 & 0.16 & 0.11 \\
Max Token Entropy & 0.37 & 0.35 & 0.19 & 0.31 & 0.15 & 0.35 & 0.19 & 0.14 \\
CCP & 0.43 & 0.43 & 0.19 & 0.33 & 0.15 & 0.38 & 0.22 & 0.17 \\
UHead & 0.51 & 0.48 & 0.21 & 0.34 & 0.24 & 0.46 & 0.31 & 0.18 \\
\bottomrule
\end{tabular}



,persons,cities,movies,inventions,books,artworks,landmarks,events
Random,0.22,0.20,0.08,0.17,0.09,0.18,0.12,0.10
MCP,0.35,0.33,0.14,0.27,0.15,0.30,0.21,0.12
Perplexity,0.32,0.28,0.13,0.24,0.11,0.32,0.16,0.11
Max Token Entropy,0.37,0.35,0.19,0.31,0.15,0.35,0.19,0.14
CCP,0.43,0.43,0.19,0.33,0.15,0.38,0.22,0.17
UHead,0.51,0.48,0.21,0.34,0.24,0.46,0.31,0.18


In [78]:
df = get_df_from_runs(
#     get_runs('test_multilang_claim', 'gemma attn'),
    {'de': '9qrlskq3', 'zh': '3vvy8fua', 'ru': '726ldk9l', 'ar': '00gsm39i'},
    splits=["ru", "zh", "de", "ar"],
    metric='pr-auc'
)
print(df.to_latex())
df

['MaxTokenEntropyClaim', 'RandomBaselineClaim', 'MaximumClaimProbability', 'LuqClaimEstimatorDummy_claim', 'CCP_claim_fact_pref', 'PerplexityClaim']
\begin{tabular}{lllll}
\toprule
 & ru & zh & de & ar \\
\midrule
Random & 0.34 & 0.23 & 0.15 & 0.29 \\
MCP & 0.43 & 0.31 & 0.20 & 0.50 \\
Perplexity & 0.40 & 0.29 & 0.15 & 0.34 \\
Max Token Entropy & 0.44 & 0.44 & 0.22 & 0.47 \\
CCP & NaN & NaN & NaN & 0.53 \\
UHead & 0.60 & 0.54 & 0.46 & 0.53 \\
\bottomrule
\end{tabular}



,ru,zh,de,ar
Random,0.34,0.23,0.15,0.29
MCP,0.43,0.31,0.20,0.50
Perplexity,0.40,0.29,0.15,0.34
Max Token Entropy,0.44,0.44,0.22,0.47
CCP,NaN,NaN,NaN,0.53
UHead,0.60,0.54,0.46,0.53


In [20]:
df = get_df_from_runs(
    {'events': 'rmq5j4yl', 'landmarks': 'ibpgyami', 'artworks': 'wp1f2lhc', 'books': '7bk3d6aq', 'inventions': '12cpoy8s', 'movies': 'cjsvgs48', 'cities': 'gy9emg9m', 'persons': 'i75hg48h'},
#     get_runs('test_multidomain_claim', 'mistral'),
    splits=["persons", "cities", "movies", "inventions", "books", "artworks", "landmarks", "events"],
    metric='pr-auc'
)
print(df.to_latex())
df

\begin{tabular}{lllllllll}
\toprule
 & persons & cities & movies & inventions & books & artworks & landmarks & events \\
\midrule
Random & 0.29 & 0.21 & 0.10 & 0.16 & 0.11 & 0.26 & 0.12 & 0.11 \\
MCP & 0.41 & 0.31 & 0.20 & 0.32 & 0.14 & 0.32 & 0.14 & 0.14 \\
Perplexity & 0.36 & 0.23 & 0.17 & 0.23 & 0.14 & 0.34 & 0.13 & 0.12 \\
Max Token Entropy & 0.42 & 0.29 & 0.24 & 0.38 & 0.17 & 0.32 & 0.14 & 0.16 \\
CCP & 0.45 & 0.35 & 0.24 & 0.35 & 0.18 & 0.37 & 0.17 & 0.15 \\
UHead & 0.63 & 0.49 & 0.33 & 0.42 & 0.32 & 0.52 & 0.32 & 0.30 \\
\bottomrule
\end{tabular}



,persons,cities,movies,inventions,books,artworks,landmarks,events
Random,0.29,0.21,0.10,0.16,0.11,0.26,0.12,0.11
MCP,0.41,0.31,0.20,0.32,0.14,0.32,0.14,0.14
Perplexity,0.36,0.23,0.17,0.23,0.14,0.34,0.13,0.12
Max Token Entropy,0.42,0.29,0.24,0.38,0.17,0.32,0.14,0.16
CCP,0.45,0.35,0.24,0.35,0.18,0.37,0.17,0.15
UHead,0.63,0.49,0.33,0.42,0.32,0.52,0.32,0.30


In [38]:
df = get_df_from_runs(
    {'events': 'peuix8vc', 'landmarks': 'xbvaicss', 'artworks': 'ifx5x00b', 'books': 'xvfzpaqy', 'inventions': 'u2e55em8', 'movies': 'aj9nmvs3', 'cities': '8yhj7nmk', 'persons': '15wrngsa'},
#     get_runs('test_multidomain_claim', 'mistral attn'),
    splits=["persons", "cities", "movies", "inventions", "books", "artworks", "landmarks", "events"],
    metric='pr-auc'
)
print(df.to_latex())
df

\begin{tabular}{lllllllll}
\toprule
 & persons & cities & movies & inventions & books & artworks & landmarks & events \\
\midrule
Random & 0.29 & 0.21 & 0.10 & 0.16 & 0.11 & 0.26 & 0.12 & 0.11 \\
MCP & 0.41 & 0.31 & 0.20 & 0.32 & 0.14 & 0.32 & 0.14 & 0.14 \\
Perplexity & 0.36 & 0.23 & 0.17 & 0.23 & 0.14 & 0.34 & 0.13 & 0.12 \\
Max Token Entropy & 0.42 & 0.29 & 0.24 & 0.38 & 0.17 & 0.32 & 0.14 & 0.16 \\
CCP & 0.45 & 0.35 & 0.24 & 0.35 & 0.18 & 0.37 & 0.17 & 0.15 \\
UHead & 0.63 & 0.45 & 0.39 & 0.48 & 0.36 & 0.53 & 0.30 & 0.30 \\
\bottomrule
\end{tabular}



,persons,cities,movies,inventions,books,artworks,landmarks,events
Random,0.29,0.21,0.10,0.16,0.11,0.26,0.12,0.11
MCP,0.41,0.31,0.20,0.32,0.14,0.32,0.14,0.14
Perplexity,0.36,0.23,0.17,0.23,0.14,0.34,0.13,0.12
Max Token Entropy,0.42,0.29,0.24,0.38,0.17,0.32,0.14,0.16
CCP,0.45,0.35,0.24,0.35,0.18,0.37,0.17,0.15
UHead,0.63,0.45,0.39,0.48,0.36,0.53,0.30,0.30


In [37]:
df = get_df_from_runs(
#     get_runs('test_multidomain_claim', 'mistral factoscope'),
    {'events': '0k34rsbw', 'landmarks': 'dpiotb3y', 'artworks': '40uteysh', 'books': 'a4nbf6lo', 'inventions': 'dxk2dp92', 'movies': 'cy7rp5jr', 'cities': 'yko0xc47', 'persons': '278wxs1l'},
    splits=["persons", "cities", "movies", "inventions", "books", "artworks", "landmarks", "events"],
    metric='pr-auc'
)
print(df.to_latex())
df

Generating results table for runs: {'events': '0k34rsbw', 'landmarks': 'dpiotb3y', 'artworks': '40uteysh', 'books': 'a4nbf6lo', 'inventions': 'dxk2dp92', 'movies': 'cy7rp5jr', 'cities': 'yko0xc47', 'persons': '278wxs1l'}
\begin{tabular}{lllllllll}
\toprule
 & persons & cities & movies & inventions & books & artworks & landmarks & events \\
\midrule
Random & 0.29 & 0.21 & 0.10 & 0.16 & 0.11 & 0.26 & 0.12 & 0.11 \\
MCP & 0.41 & 0.31 & 0.20 & 0.32 & 0.14 & 0.32 & 0.14 & 0.14 \\
Perplexity & 0.36 & 0.23 & 0.17 & 0.23 & 0.14 & 0.34 & 0.13 & 0.12 \\
Max Token Entropy & 0.42 & 0.29 & 0.24 & 0.38 & 0.17 & 0.32 & 0.14 & 0.16 \\
CCP & 0.45 & 0.35 & 0.24 & 0.35 & 0.18 & 0.37 & 0.17 & 0.15 \\
UHead & 0.61 & 0.47 & 0.34 & 0.42 & 0.32 & 0.49 & 0.28 & 0.26 \\
\bottomrule
\end{tabular}



,persons,cities,movies,inventions,books,artworks,landmarks,events
Random,0.29,0.21,0.10,0.16,0.11,0.26,0.12,0.11
MCP,0.41,0.31,0.20,0.32,0.14,0.32,0.14,0.14
Perplexity,0.36,0.23,0.17,0.23,0.14,0.34,0.13,0.12
Max Token Entropy,0.42,0.29,0.24,0.38,0.17,0.32,0.14,0.16
CCP,0.45,0.35,0.24,0.35,0.18,0.37,0.17,0.15
UHead,0.61,0.47,0.34,0.42,0.32,0.49,0.28,0.26


In [19]:
df = get_df_from_runs(
    {'events': 'uhzc6p3n', 'landmarks': '7674oy2c', 'artworks': 'zzzexkck', 'books': '44pyh719', 'inventions': 'c3zcv8kz', 'movies': 'k6x93v9v', 'cities': '7ejgeyhp', 'persons': 'o03lkeul'},
#     get_runs('test_multidomain_claim', 'gemma'),
    splits=["persons", "cities", "movies", "inventions", "books", "artworks", "landmarks", "events"],
    metric='pr-auc'
)
print(df.to_latex())
df

\begin{tabular}{lllllllll}
\toprule
 & persons & cities & movies & inventions & books & artworks & landmarks & events \\
\midrule
Random & 0.13 & 0.12 & 0.06 & 0.08 & 0.04 & 0.15 & 0.06 & 0.05 \\
MCP & 0.18 & 0.18 & 0.07 & 0.13 & 0.06 & 0.22 & 0.12 & 0.06 \\
Perplexity & 0.14 & 0.14 & 0.09 & 0.11 & 0.05 & 0.24 & 0.09 & 0.07 \\
Max Token Entropy & 0.20 & 0.16 & 0.11 & 0.16 & 0.08 & 0.26 & 0.11 & 0.08 \\
CCP & 0.27 & 0.21 & 0.12 & 0.17 & 0.06 & 0.22 & 0.12 & 0.07 \\
UHead & 0.43 & 0.33 & 0.26 & 0.24 & 0.12 & 0.42 & 0.23 & 0.13 \\
\bottomrule
\end{tabular}



,persons,cities,movies,inventions,books,artworks,landmarks,events
Random,0.13,0.12,0.06,0.08,0.04,0.15,0.06,0.05
MCP,0.18,0.18,0.07,0.13,0.06,0.22,0.12,0.06
Perplexity,0.14,0.14,0.09,0.11,0.05,0.24,0.09,0.07
Max Token Entropy,0.20,0.16,0.11,0.16,0.08,0.26,0.11,0.08
CCP,0.27,0.21,0.12,0.17,0.06,0.22,0.12,0.07
UHead,0.43,0.33,0.26,0.24,0.12,0.42,0.23,0.13


In [133]:
df = get_df_from_runs(
    {'events': 'emyfvbbc', 'landmarks': '8wpt7qcv', 'artworks': '650nr5rv', 'books': 'xzpo9uju', 'inventions': 'wqyu7thj', 'movies': '47sbspxz', 'cities': 'cy1m1v6t', 'persons': '5m231irc'},
#     get_runs('test_multidomain_claim', 'gemma saplma'),
    splits=["persons", "cities", "movies", "inventions", "books", "artworks", "landmarks", "events"],
    metric='pr-auc'
)
print(df.to_latex())
df

Generating results table for runs: {'events': 'emyfvbbc', 'landmarks': '8wpt7qcv', 'artworks': '650nr5rv', 'books': 'xzpo9uju', 'inventions': 'wqyu7thj', 'movies': '47sbspxz', 'cities': 'cy1m1v6t', 'persons': '5m231irc'}
\begin{tabular}{lllllllll}
\toprule
 & persons & cities & movies & inventions & books & artworks & landmarks & events \\
\midrule
Random & 0.13 & 0.12 & 0.04 & 0.09 & 0.04 & 0.14 & 0.07 & 0.06 \\
MCP & 0.19 & 0.17 & 0.07 & 0.16 & 0.07 & 0.22 & 0.12 & 0.06 \\
Perplexity & 0.14 & 0.12 & 0.08 & 0.11 & 0.06 & 0.25 & 0.08 & 0.07 \\
Max Token Entropy & 0.20 & 0.14 & 0.09 & 0.16 & 0.08 & 0.27 & 0.10 & 0.10 \\
UHead (mean prob., bio) & 0.34 & 0.25 & 0.18 & 0.22 & 0.12 & 0.33 & 0.15 & 0.13 \\
UHead (mean logits, bio) & 0.34 & 0.26 & 0.20 & 0.23 & 0.12 & 0.33 & 0.15 & 0.13 \\
UHead (prob. prod., bio) & 0.23 & 0.14 & 0.11 & 0.13 & 0.07 & 0.26 & 0.10 & 0.11 \\
\bottomrule
\end{tabular}



,persons,cities,movies,inventions,books,artworks,landmarks,events
Random,0.13,0.12,0.04,0.09,0.04,0.14,0.07,0.06
MCP,0.19,0.17,0.07,0.16,0.07,0.22,0.12,0.06
Perplexity,0.14,0.12,0.08,0.11,0.06,0.25,0.08,0.07
Max Token Entropy,0.20,0.14,0.09,0.16,0.08,0.27,0.10,0.10
"UHead (mean prob., bio)",0.34,0.25,0.18,0.22,0.12,0.33,0.15,0.13
"UHead (mean logits, bio)",0.34,0.26,0.20,0.23,0.12,0.33,0.15,0.13
"UHead (prob. prod., bio)",0.23,0.14,0.11,0.13,0.07,0.26,0.10,0.11


In [135]:
df = get_df_from_runs(
#     get_runs('test_multidomain_claim', 'mistral lblens'),
    {'events': 'k29j6kp2', 'landmarks': '70gjmmdb', 'artworks': '1912ptmt', 'books': 'n8t77wzu', 'inventions': 'cdr4n8na', 'movies': '5ijabeer', 'cities': 'un2t458h', 'persons': 'pksvls7j'},
    splits=["persons", "cities", "movies", "inventions", "books", "artworks", "landmarks", "events"],
    metric='pr-auc'
)
print(df.to_latex())
df

Generating results table for runs: {'events': 'k29j6kp2', 'landmarks': '70gjmmdb', 'artworks': '1912ptmt', 'books': 'n8t77wzu', 'inventions': 'cdr4n8na', 'movies': '5ijabeer', 'cities': 'un2t458h', 'persons': 'pksvls7j'}
\begin{tabular}{lllllllll}
\toprule
 & persons & cities & movies & inventions & books & artworks & landmarks & events \\
\midrule
Random & 0.29 & 0.21 & 0.10 & 0.16 & 0.11 & 0.26 & 0.12 & 0.11 \\
MCP & 0.41 & 0.31 & 0.20 & 0.32 & 0.14 & 0.32 & 0.14 & 0.14 \\
Perplexity & 0.36 & 0.23 & 0.17 & 0.23 & 0.14 & 0.34 & 0.13 & 0.12 \\
Max Token Entropy & 0.42 & 0.29 & 0.24 & 0.38 & 0.17 & 0.32 & 0.14 & 0.16 \\
UHead & 0.56 & 0.45 & 0.25 & 0.39 & 0.26 & 0.46 & 0.26 & 0.29 \\
\bottomrule
\end{tabular}



,persons,cities,movies,inventions,books,artworks,landmarks,events
Random,0.29,0.21,0.10,0.16,0.11,0.26,0.12,0.11
MCP,0.41,0.31,0.20,0.32,0.14,0.32,0.14,0.14
Perplexity,0.36,0.23,0.17,0.23,0.14,0.34,0.13,0.12
Max Token Entropy,0.42,0.29,0.24,0.38,0.17,0.32,0.14,0.16
UHead,0.56,0.45,0.25,0.39,0.26,0.46,0.26,0.29


In [161]:
df = get_df_from_runs(
    {'events': 'xwqwcf22', 'landmarks': '65jmea7j', 'artworks': 'ns5635fg', 'books': '55jt867w', 'inventions': '7f4ppjgd', 'movies': 'cya6dbv0', 'cities': '9mbwodo5', 'persons': 'vzd4rbs4'},
#     get_runs('test_multidomain', 'mistral ccp'),
    splits=["persons", "cities", "movies", "inventions", "books", "artworks", "landmarks", "events"],
    metric='pr-auc'
)
print(df.to_latex())
df

\begin{tabular}{lllllllll}
\toprule
 & persons & cities & movies & inventions & books & artworks & landmarks & events \\
\midrule
Random & 0.29 & 0.21 & 0.10 & 0.16 & 0.11 & 0.26 & 0.12 & 0.11 \\
MCP & 0.41 & 0.31 & 0.20 & 0.32 & 0.14 & 0.32 & 0.14 & 0.14 \\
Perplexity & 0.36 & 0.23 & 0.17 & 0.23 & 0.14 & 0.34 & 0.13 & 0.12 \\
Max Token Entropy & 0.42 & 0.29 & 0.24 & 0.38 & 0.17 & 0.32 & 0.14 & 0.16 \\
CCP_fact_pref & 0.50 & 0.37 & 0.27 & 0.38 & 0.17 & 0.38 & 0.20 & 0.17 \\
UHead & 0.56 & 0.45 & 0.25 & 0.39 & 0.26 & 0.46 & 0.26 & 0.29 \\
\bottomrule
\end{tabular}



,persons,cities,movies,inventions,books,artworks,landmarks,events
Random,0.29,0.21,0.10,0.16,0.11,0.26,0.12,0.11
MCP,0.41,0.31,0.20,0.32,0.14,0.32,0.14,0.14
Perplexity,0.36,0.23,0.17,0.23,0.14,0.34,0.13,0.12
Max Token Entropy,0.42,0.29,0.24,0.38,0.17,0.32,0.14,0.16
CCP_fact_pref,0.50,0.37,0.27,0.38,0.17,0.38,0.20,0.17
UHead,0.56,0.45,0.25,0.39,0.26,0.46,0.26,0.29


In [162]:
df = get_df_from_runs(
    {'events': '97dz1o0b', 'landmarks': '8avgsq1d', 'artworks': 'os7pwlpm', 'books': 'vyn1gb7r', 'inventions': 'etjf5uzv', 'movies': 'uxx7lams', 'cities': 'd90y2sm1', 'persons': 'se43uxc4'},
#     get_runs('test_multidomain', 'gemma ccp'),
    splits=["persons", "cities", "movies", "inventions", "books", "artworks", "landmarks", "events"],
    metric='pr-auc'
)
print(df.to_latex())
df

\begin{tabular}{lllllllll}
\toprule
 & persons & cities & movies & inventions & books & artworks & landmarks & events \\
\midrule
Random & 0.13 & 0.12 & 0.06 & 0.08 & 0.04 & 0.15 & 0.06 & 0.05 \\
MCP & 0.18 & 0.18 & 0.07 & 0.13 & 0.06 & 0.22 & 0.12 & 0.06 \\
Perplexity & 0.14 & 0.14 & 0.09 & 0.11 & 0.05 & 0.24 & 0.09 & 0.07 \\
Max Token Entropy & 0.20 & 0.16 & 0.11 & 0.16 & 0.08 & 0.26 & 0.11 & 0.08 \\
CCP_fact_pref & 0.31 & 0.23 & 0.17 & 0.16 & 0.07 & 0.22 & 0.10 & 0.07 \\
UHead & 0.13 & 0.13 & 0.05 & 0.08 & 0.04 & 0.15 & 0.06 & 0.05 \\
\bottomrule
\end{tabular}



,persons,cities,movies,inventions,books,artworks,landmarks,events
Random,0.13,0.12,0.06,0.08,0.04,0.15,0.06,0.05
MCP,0.18,0.18,0.07,0.13,0.06,0.22,0.12,0.06
Perplexity,0.14,0.14,0.09,0.11,0.05,0.24,0.09,0.07
Max Token Entropy,0.20,0.16,0.11,0.16,0.08,0.26,0.11,0.08
CCP_fact_pref,0.31,0.23,0.17,0.16,0.07,0.22,0.10,0.07
UHead,0.13,0.13,0.05,0.08,0.04,0.15,0.06,0.05


In [163]:
df = get_df_from_runs(
    {'de': 'wlzgdvxz', 'zh': '7wf878gt', 'ru': '8ajorbrv'},
#     get_runs('test_multilang_claim', 'gemma ccp'),
    splits=["ru", "zh", "de"],
    metric='pr-auc'
)
print(df.to_latex())
df

\begin{tabular}{llll}
\toprule
 & ru & zh & de \\
\midrule
Random & 0.34 & 0.23 & 0.15 \\
MCP & 0.43 & 0.31 & 0.20 \\
Perplexity & 0.40 & 0.29 & 0.15 \\
Max Token Entropy & 0.44 & 0.44 & 0.22 \\
CCP_fact_pref & 0.49 & 0.44 & 0.31 \\
UHead & 0.33 & 0.22 & 0.15 \\
\bottomrule
\end{tabular}



,ru,zh,de
Random,0.34,0.23,0.15
MCP,0.43,0.31,0.20
Perplexity,0.40,0.29,0.15
Max Token Entropy,0.44,0.44,0.22
CCP_fact_pref,0.49,0.44,0.31
UHead,0.33,0.22,0.15


In [3]:
df = get_df_from_runs(
#     {'de': '940suw74', 'zh': 'kryiw4ql', 'ru': 'a4kygdvh'},  # with bug
    {'de': 'kfg3pzl7', 'zh': 'ykry00f6', 'ru': '87la8g08'},  # without bug
#     get_runs('test_multilang_claim', 'gemma lblens'),
    splits=["ru", "zh", "de"],
    metric='pr-auc'
)
print(df.to_latex())
df

Generating results table for runs: {'de': 'kfg3pzl7', 'zh': 'ykry00f6', 'ru': '87la8g08'}
\begin{tabular}{llll}
\toprule
 & ru & zh & de \\
\midrule
Random & 0.34 & 0.23 & 0.15 \\
MCP & 0.43 & 0.31 & 0.20 \\
Perplexity & 0.40 & 0.29 & 0.15 \\
Max Token Entropy & 0.44 & 0.44 & 0.22 \\
UHead & 0.58 & 0.48 & 0.39 \\
\bottomrule
\end{tabular}



,ru,zh,de
Random,0.34,0.23,0.15
MCP,0.43,0.31,0.20
Perplexity,0.40,0.29,0.15
Max Token Entropy,0.44,0.44,0.22
UHead,0.58,0.48,0.39


In [5]:
df = get_df_from_runs(
#     {'events': 'hpa2uyki', 'landmarks': 'pn0zzkvi', 'artworks': '4v81wrl7', 'books': '1jvrt3fw', 'inventions': 'n85j03zj', 'movies': 'aah79ocf', 'cities': '4vwsge6v', 'persons': 't6up3rmg'},  # with bug
    {'events': '9hvgpu50', 'landmarks': '625qpm99', 'artworks': 'r3dua7ef', 'books': 'cqaaeu9r', 'inventions': 'zi4wlcri', 'movies': 'q9ckoa2w', 'cities': 'xdhqfwr3', 'persons': 'r5dk6tfj'},  # without bugs
#     get_runs('test_multidomain_claim', 'gemma lblens'),
    splits=["persons", "cities", "movies", "inventions", "books", "artworks", "landmarks", "events"],
    metric='pr-auc'
)
print(df.to_latex())
df

Generating results table for runs: {'events': '9hvgpu50', 'landmarks': '625qpm99', 'artworks': 'r3dua7ef', 'books': 'cqaaeu9r', 'inventions': 'zi4wlcri', 'movies': 'q9ckoa2w', 'cities': 'xdhqfwr3', 'persons': 'r5dk6tfj'}
\begin{tabular}{llllllll}
\toprule
 & persons & cities & movies & inventions & books & artworks & landmarks \\
\midrule
Random & 0.13 & 0.12 & 0.06 & 0.08 & 0.04 & 0.15 & 0.06 \\
MCP & 0.18 & 0.18 & 0.07 & 0.13 & 0.06 & 0.22 & 0.12 \\
Perplexity & 0.14 & 0.14 & 0.09 & 0.11 & 0.05 & 0.24 & 0.09 \\
Max Token Entropy & 0.20 & 0.16 & 0.11 & 0.16 & 0.08 & 0.26 & 0.11 \\
UHead & 0.36 & 0.29 & 0.24 & 0.21 & 0.10 & 0.30 & 0.12 \\
\bottomrule
\end{tabular}



,persons,cities,movies,inventions,books,artworks,landmarks
Random,0.13,0.12,0.06,0.08,0.04,0.15,0.06
MCP,0.18,0.18,0.07,0.13,0.06,0.22,0.12
Perplexity,0.14,0.14,0.09,0.11,0.05,0.24,0.09
Max Token Entropy,0.20,0.16,0.11,0.16,0.08,0.26,0.11
UHead,0.36,0.29,0.24,0.21,0.10,0.30,0.12


In [152]:
df = get_df_from_runs(
    {'events': 'vdw8no5q', 'landmarks': 'sjls7cbj', 'artworks': 'kn08tkvn', 'books': 'b0m14m8s', 'inventions': 'o50q8ed3', 'movies': '6lbmxy35', 'cities': '3s91bo6y', 'persons': 'o8lnqlk0'},
#     get_runs('test_multidomain_claim', 'gemma attn'),
    splits=["persons", "cities", "movies", "inventions", "books", "artworks", "landmarks", "events"],
    metric='pr-auc'
)
print(df.to_latex())
df

wandb: WARNING A graphql request initiated by the public wandb API timed out (timeout=9 sec). Create a new API with an integer timeout larger than 9, e.g., `api = wandb.Api(timeout=19)` to increase the graphql timeout.


Generating results table for runs: {'events': 'vdw8no5q', 'landmarks': 'sjls7cbj', 'artworks': 'kn08tkvn', 'books': 'b0m14m8s', 'inventions': 'o50q8ed3', 'movies': '6lbmxy35', 'cities': '3s91bo6y', 'persons': 'o8lnqlk0'}
\begin{tabular}{lllllllll}
\toprule
 & persons & cities & movies & inventions & books & artworks & landmarks & events \\
\midrule
Random & 0.13 & 0.12 & 0.06 & 0.08 & 0.04 & 0.15 & 0.06 & 0.05 \\
MCP & 0.18 & 0.18 & 0.07 & 0.13 & 0.06 & 0.22 & 0.12 & 0.06 \\
Perplexity & 0.14 & 0.14 & 0.09 & 0.11 & 0.05 & 0.24 & 0.09 & 0.07 \\
Max Token Entropy & 0.20 & 0.16 & 0.11 & 0.16 & 0.08 & 0.26 & 0.11 & 0.08 \\
UHead & 0.44 & 0.26 & 0.27 & 0.22 & 0.08 & 0.32 & 0.18 & 0.12 \\
\bottomrule
\end{tabular}



,persons,cities,movies,inventions,books,artworks,landmarks,events
Random,0.13,0.12,0.06,0.08,0.04,0.15,0.06,0.05
MCP,0.18,0.18,0.07,0.13,0.06,0.22,0.12,0.06
Perplexity,0.14,0.14,0.09,0.11,0.05,0.24,0.09,0.07
Max Token Entropy,0.20,0.16,0.11,0.16,0.08,0.26,0.11,0.08
UHead,0.44,0.26,0.27,0.22,0.08,0.32,0.18,0.12


In [132]:
df = get_df_from_runs(
    {'events': 'dq3tvj9y', 'landmarks': 'jsy6cbxx', 'artworks': '1pol7x3g', 'books': 'fd1ssvtg', 'inventions': 'yyfqpy4t', 'movies': 't8itvscc', 'cities': 's50r5qji'},
#     get_runs('test_multidomain_claim', 'mistral attn bio+all-1'),
    splits=["cities", "movies", "inventions", "books", "artworks", "landmarks", "events"],
    metric='pr-auc'
)
print(df.to_latex())
df

\begin{tabular}{llllllll}
\toprule
 & cities & movies & inventions & books & artworks & landmarks & events \\
\midrule
Random & 0.21 & 0.10 & 0.16 & 0.11 & 0.26 & 0.12 & 0.11 \\
MCP & 0.31 & 0.20 & 0.32 & 0.14 & 0.32 & 0.14 & 0.14 \\
Perplexity & 0.23 & 0.17 & 0.23 & 0.14 & 0.34 & 0.13 & 0.12 \\
Max Token Entropy & 0.29 & 0.24 & 0.38 & 0.17 & 0.32 & 0.14 & 0.16 \\
UHead & 0.49 & 0.42 & 0.49 & 0.39 & 0.55 & 0.31 & 0.35 \\
\bottomrule
\end{tabular}



,cities,movies,inventions,books,artworks,landmarks,events
Random,0.21,0.10,0.16,0.11,0.26,0.12,0.11
MCP,0.31,0.20,0.32,0.14,0.32,0.14,0.14
Perplexity,0.23,0.17,0.23,0.14,0.34,0.13,0.12
Max Token Entropy,0.29,0.24,0.38,0.17,0.32,0.14,0.16
UHead,0.49,0.42,0.49,0.39,0.55,0.31,0.35


In [18]:
df = get_df_from_runs(
    {'de': 'ay8eojx5', 'zh': 'tsu5pyek', 'ru': 'k5hge2wp', 'en': 'sboe48ib'},
#     get_runs('test_multilang_claim', 'gemma'),
    splits=["en", "ru", "zh", "de"],
    metric='pr-auc'
)
print(df.to_latex())
df

Generating results table for runs: {'de': 'ay8eojx5', 'zh': 'tsu5pyek', 'ru': 'k5hge2wp', 'en': 'sboe48ib'}
\begin{tabular}{lllll}
\toprule
 & en & ru & zh & de \\
\midrule
Random & 0.05 & 0.34 & 0.23 & 0.15 \\
MCP & 0.09 & 0.43 & 0.31 & 0.20 \\
Perplexity & 0.06 & 0.40 & 0.29 & 0.15 \\
Max Token Entropy & 0.08 & 0.44 & 0.44 & 0.22 \\
CCP & 0.08 & 0.49 & 0.37 & 0.26 \\
UHead & 0.28 & 0.58 & 0.56 & 0.46 \\
\bottomrule
\end{tabular}



,en,ru,zh,de
Random,0.05,0.34,0.23,0.15
MCP,0.09,0.43,0.31,0.20
Perplexity,0.06,0.40,0.29,0.15
Max Token Entropy,0.08,0.44,0.44,0.22
CCP,0.08,0.49,0.37,0.26
UHead,0.28,0.58,0.56,0.46


In [169]:
df = get_df_from_runs(
    {'events': '38x5vdbn', 'landmarks': 'ttv3a8vf', 'artworks': '82fajhd8', 'books': '7pfrw35x', 'inventions': 'sjagh58p', 'movies': 'emxjx8br', 'cities': 'ad6xq7o8', 'persons': 'yv29higf'},
#     get_runs('test_multidomain', 'mistral'),
    splits=["persons", "cities", "movies", "inventions", "books", "artworks", "landmarks", "events"],
    metric='pr-auc'
)
print(df.to_latex())
df

\begin{tabular}{lllllllll}
\toprule
 & persons & cities & movies & inventions & books & artworks & landmarks & events \\
\midrule
Random & 0.25 & 0.20 & 0.10 & 0.13 & 0.09 & 0.25 & 0.12 & 0.12 \\
MCP & 0.33 & 0.31 & 0.16 & 0.26 & 0.14 & 0.31 & 0.16 & 0.13 \\
Perplexity & 0.30 & 0.23 & 0.15 & 0.22 & 0.12 & 0.34 & 0.14 & 0.11 \\
Max Token Entropy & 0.35 & 0.28 & 0.17 & 0.33 & 0.15 & 0.33 & 0.14 & 0.13 \\
CCP & 0.39 & 0.32 & 0.18 & 0.28 & 0.16 & 0.37 & 0.20 & 0.15 \\
UHead (mean prob., bio) & 0.52 & 0.44 & 0.30 & 0.34 & 0.27 & 0.56 & 0.34 & 0.26 \\
UHead (mean logits, bio) & 0.52 & 0.43 & 0.31 & 0.34 & 0.28 & 0.56 & 0.34 & 0.27 \\
UHead (prob. prod., bio) & 0.43 & 0.29 & 0.22 & 0.21 & 0.20 & 0.50 & 0.28 & 0.16 \\
\bottomrule
\end{tabular}



,persons,cities,movies,inventions,books,artworks,landmarks,events
Random,0.25,0.20,0.10,0.13,0.09,0.25,0.12,0.12
MCP,0.33,0.31,0.16,0.26,0.14,0.31,0.16,0.13
Perplexity,0.30,0.23,0.15,0.22,0.12,0.34,0.14,0.11
Max Token Entropy,0.35,0.28,0.17,0.33,0.15,0.33,0.14,0.13
CCP,0.39,0.32,0.18,0.28,0.16,0.37,0.20,0.15
"UHead (mean prob., bio)",0.52,0.44,0.30,0.34,0.27,0.56,0.34,0.26
"UHead (mean logits, bio)",0.52,0.43,0.31,0.34,0.28,0.56,0.34,0.27
"UHead (prob. prod., bio)",0.43,0.29,0.22,0.21,0.20,0.50,0.28,0.16


In [170]:
df = get_df_from_runs(
    {'events': '3e0mjkoz', 'landmarks': 'v895fjtu', 'artworks': '10n2tw64', 'books': 'ix0gvutj', 'inventions': 'gg83lf7z', 'movies': '2m8wztcm', 'cities': 'yzwe5h4o'},
#     get_runs('test_multidomain', 'mistral bio+all-1'),
    splits=["persons", "cities", "movies", "inventions", "books", "artworks", "landmarks", "events"],
    metric='pr-auc'
)
print(df.to_latex())
df

\begin{tabular}{llllllll}
\toprule
 & cities & movies & inventions & books & artworks & landmarks & events \\
\midrule
Random & 0.20 & 0.10 & 0.13 & 0.09 & 0.25 & 0.12 & 0.12 \\
MCP & 0.31 & 0.16 & 0.26 & 0.14 & 0.31 & 0.16 & 0.13 \\
Perplexity & 0.23 & 0.15 & 0.22 & 0.12 & 0.34 & 0.14 & 0.11 \\
Max Token Entropy & 0.28 & 0.17 & 0.33 & 0.15 & 0.33 & 0.14 & 0.13 \\
CCP & 0.32 & 0.18 & 0.28 & 0.16 & 0.37 & 0.20 & 0.15 \\
UHead (mean prob., bio) & 0.43 & 0.32 & 0.34 & 0.27 & 0.56 & 0.35 & 0.26 \\
UHead (mean logits, bio) & 0.43 & 0.33 & 0.34 & 0.27 & 0.56 & 0.36 & 0.27 \\
UHead (prob. prod., bio) & 0.31 & 0.23 & 0.22 & 0.19 & 0.50 & 0.31 & 0.18 \\
\bottomrule
\end{tabular}



,cities,movies,inventions,books,artworks,landmarks,events
Random,0.20,0.10,0.13,0.09,0.25,0.12,0.12
MCP,0.31,0.16,0.26,0.14,0.31,0.16,0.13
Perplexity,0.23,0.15,0.22,0.12,0.34,0.14,0.11
Max Token Entropy,0.28,0.17,0.33,0.15,0.33,0.14,0.13
CCP,0.32,0.18,0.28,0.16,0.37,0.20,0.15
"UHead (mean prob., bio)",0.43,0.32,0.34,0.27,0.56,0.35,0.26
"UHead (mean logits, bio)",0.43,0.33,0.34,0.27,0.56,0.36,0.27
"UHead (prob. prod., bio)",0.31,0.23,0.22,0.19,0.50,0.31,0.18


In [171]:
df = get_df_from_runs(
    {'events': 'adsvvvhj', 'landmarks': 'gkonsx03', 'artworks': 'x1avjr2r', 'books': 'pqk0gy9b', 'inventions': 'zv13n06a', 'movies': 'yiz57nm3', 'cities': 'vbekhrje', 'persons': '9dfm4h04'},
#     get_runs('test_multidomain', 'mistral saplma'),
    splits=["persons", "cities", "movies", "inventions", "books", "artworks", "landmarks", "events"],
    metric='pr-auc'
)
print(df.to_latex())
df

\begin{tabular}{lllllllll}
\toprule
 & persons & cities & movies & inventions & books & artworks & landmarks & events \\
\midrule
Random & 0.25 & 0.20 & 0.10 & 0.13 & 0.09 & 0.25 & 0.12 & 0.12 \\
MCP & 0.33 & 0.31 & 0.16 & 0.26 & 0.14 & 0.31 & 0.16 & 0.13 \\
Perplexity & 0.30 & 0.23 & 0.15 & 0.22 & 0.12 & 0.34 & 0.14 & 0.11 \\
Max Token Entropy & 0.35 & 0.28 & 0.17 & 0.33 & 0.15 & 0.33 & 0.14 & 0.13 \\
CCP & 0.39 & 0.32 & 0.18 & 0.28 & 0.16 & 0.37 & 0.20 & 0.15 \\
UHead (mean prob., bio) & 0.54 & 0.43 & 0.27 & 0.35 & 0.29 & 0.53 & 0.35 & 0.24 \\
UHead (mean logits, bio) & 0.53 & 0.43 & 0.26 & 0.35 & 0.29 & 0.53 & 0.35 & 0.24 \\
UHead (prob. prod., bio) & 0.44 & 0.29 & 0.21 & 0.22 & 0.21 & 0.47 & 0.30 & 0.15 \\
\bottomrule
\end{tabular}



,persons,cities,movies,inventions,books,artworks,landmarks,events
Random,0.25,0.20,0.10,0.13,0.09,0.25,0.12,0.12
MCP,0.33,0.31,0.16,0.26,0.14,0.31,0.16,0.13
Perplexity,0.30,0.23,0.15,0.22,0.12,0.34,0.14,0.11
Max Token Entropy,0.35,0.28,0.17,0.33,0.15,0.33,0.14,0.13
CCP,0.39,0.32,0.18,0.28,0.16,0.37,0.20,0.15
"UHead (mean prob., bio)",0.54,0.43,0.27,0.35,0.29,0.53,0.35,0.24
"UHead (mean logits, bio)",0.53,0.43,0.26,0.35,0.29,0.53,0.35,0.24
"UHead (prob. prod., bio)",0.44,0.29,0.21,0.22,0.21,0.47,0.30,0.15


In [182]:
df = get_df_from_runs(
    {'de': 'q27nhpi0', 'en': '8w6fw5tc', 'ar': '3md4e5j1', 'zh': '883aqt39', 'ru': 'zes8mmlp'},
#     get_runs('test_multilang', 'gemma'),
    splits=["en", "ru", "zh", "ar", "de"],
    metric='pr-auc'
)
print(df.to_latex())
df

\begin{tabular}{llllll}
\toprule
 & en & ru & zh & ar & de \\
\midrule
Random & 0.05 & 0.34 & 0.23 & 0.23 & 0.15 \\
MCP & 0.09 & 0.43 & 0.31 & 0.42 & 0.20 \\
Perplexity & 0.06 & 0.40 & 0.29 & 0.30 & 0.15 \\
Max Token Entropy & 0.08 & 0.44 & 0.44 & 0.41 & 0.22 \\
CCP & 0.08 & 0.49 & 0.37 & 0.48 & 0.26 \\
UHead (mean prob., bio) & 0.20 & 0.58 & 0.48 & 0.40 & 0.40 \\
UHead (mean logits, bio) & 0.20 & 0.58 & 0.48 & 0.40 & 0.40 \\
UHead (prob. prod., bio) & 0.09 & 0.47 & 0.36 & 0.27 & 0.22 \\
\bottomrule
\end{tabular}



,en,ru,zh,ar,de
Random,0.05,0.34,0.23,0.23,0.15
MCP,0.09,0.43,0.31,0.42,0.20
Perplexity,0.06,0.40,0.29,0.30,0.15
Max Token Entropy,0.08,0.44,0.44,0.41,0.22
CCP,0.08,0.49,0.37,0.48,0.26
"UHead (mean prob., bio)",0.20,0.58,0.48,0.40,0.40
"UHead (mean logits, bio)",0.20,0.58,0.48,0.40,0.40
"UHead (prob. prod., bio)",0.09,0.47,0.36,0.27,0.22


In [184]:
df = get_df_from_runs(
#     get_runs('test_multilang', 'saplma'),
    {'de': 'ek9d11ky', 'ar': 'a44rebsz', 'zh': 't8m7vvzf', 'ru': 'fqwddfd6', 'en': 'lny67fzm'},
    splits=["en", "ru", "zh", "ar", "de"],
    metric='pr-auc'
)
print(df.to_latex())
df

Generating results table for runs: {'de': 'ek9d11ky', 'ar': 'a44rebsz', 'zh': 't8m7vvzf', 'ru': 'fqwddfd6', 'en': 'lny67fzm'}
\begin{tabular}{llllll}
\toprule
 & en & ru & zh & ar & de \\
\midrule
Random & 0.05 & 0.34 & 0.23 & 0.23 & 0.15 \\
MCP & 0.09 & 0.43 & 0.31 & 0.42 & 0.20 \\
Perplexity & 0.06 & 0.40 & 0.29 & 0.30 & 0.15 \\
Max Token Entropy & 0.08 & 0.44 & 0.44 & 0.41 & 0.22 \\
CCP & 0.08 & 0.49 & 0.37 & 0.48 & 0.26 \\
UHead (mean prob., bio) & 0.26 & 0.51 & 0.33 & 0.37 & 0.39 \\
UHead (mean logits, bio) & 0.27 & 0.53 & 0.35 & 0.36 & 0.39 \\
UHead (prob. prod., bio) & 0.11 & 0.39 & 0.23 & 0.23 & 0.20 \\
\bottomrule
\end{tabular}



,en,ru,zh,ar,de
Random,0.05,0.34,0.23,0.23,0.15
MCP,0.09,0.43,0.31,0.42,0.20
Perplexity,0.06,0.40,0.29,0.30,0.15
Max Token Entropy,0.08,0.44,0.44,0.41,0.22
CCP,0.08,0.49,0.37,0.48,0.26
"UHead (mean prob., bio)",0.26,0.51,0.33,0.37,0.39
"UHead (mean logits, bio)",0.27,0.53,0.35,0.36,0.39
"UHead (prob. prod., bio)",0.11,0.39,0.23,0.23,0.20


In [79]:
some_mistral_runs = {'events': '3e0mjkoz', 'landmarks': 'v895fjtu', 'artworks': '10n2tw64', 'books': 'ix0gvutj', 'inventions': 'gg83lf7z', 'movies': '2m8wztcm', 'cities': 'yzwe5h4o'}
some_gemma_runs = {'events': 'uhzc6p3n', 'landmarks': '7674oy2c', 'artworks': 'zzzexkck', 'books': '44pyh719', 'inventions': 'c3zcv8kz', 'movies': 'k6x93v9v', 'cities': '7ejgeyhp', 'persons': 'o03lkeul'}

run_id = some_mistral_runs['events']

def get_logs(run_id):
    run = wandb.Api().run(f"nlpresearch.group/luq/{run_id}")
    for file in run.files():
        if "output.log" in file.name:
            file.download(replace=True)
            with open(file.name, "r") as f:
                logs = f.read()
    return logs

In [81]:
def get_n_claims(run_id):
    logs = get_logs(run_id)
    claims = 0
    for i, l in enumerate(logs.split('\n')):
        spec = " - ('claim', 'RandomBaselineClaim'): ["
        if spec in l:
            baselines = l.split(spec, 1)[-1][:-1]
            claims += len(baselines.split(', '))
    return claims 

splits = ['persons', 'cities', 'movies', 'inventions', 'books', 'artworks', 'landmarks', 'events']
some_mistral_runs = {'events': 'rmq5j4yl', 'landmarks': 'ibpgyami', 'artworks': 'wp1f2lhc', 'books': '7bk3d6aq', 'inventions': '12cpoy8s', 'movies': 'cjsvgs48', 'cities': 'gy9emg9m', 'persons': 'i75hg48h'}
some_gemma_runs = {'events': 'uhzc6p3n', 'landmarks': '7674oy2c', 'artworks': 'zzzexkck', 'books': '44pyh719', 'inventions': 'c3zcv8kz', 'movies': 'k6x93v9v', 'cities': '7ejgeyhp', 'persons': 'o03lkeul'}
# splits = ['en', 'ru', 'zh', 'de']
# some_mistral_runs = {'de': 'ay8eojx5', 'zh': 'tsu5pyek', 'ru': 'k5hge2wp', 'en': 'sboe48ib'}
# some_gemma_runs = {'de': 'ay8eojx5', 'zh': 'tsu5pyek', 'ru': 'k5hge2wp', 'en': 'sboe48ib'}

for split in splits:
    print(split, end=' ')
    for runs in [some_mistral_runs, some_gemma_runs]:
        claims = get_n_claims(runs[split])
        print('&', claims, end=' ')
    print()

persons & 2234 & 2857 
cities & 2128 & 2684 
movies & 2568 & 3121 
inventions & 2269 & 2626 
books & 2530 & 3070 
artworks & 2464 & 2873 
landmarks & 2365 & 2566 
events & 2294 & 2665 


In [88]:
def get_base_accuracy(run_id):
    logs = get_logs(run_id)
    claims = 0
    fact_checks = []
    for i, l in enumerate(logs.split('\n')):
        spec = " - ('claim', 'OpenAIFactCheck'): ["
        if spec in l:
            baselines = l.split(spec, 1)[-1][:-1]
            for c in baselines.split(', '):
                if c != 'nan':
                    fact_checks.append(int(c))
    return str(round(100 * (1 - np.mean(fact_checks)), 1)) + '\%'

# splits = ['persons', 'cities', 'movies', 'inventions', 'books', 'artworks', 'landmarks', 'events']
# some_mistral_runs = {'events': 'rmq5j4yl', 'landmarks': 'ibpgyami', 'artworks': 'wp1f2lhc', 'books': '7bk3d6aq', 'inventions': '12cpoy8s', 'movies': 'cjsvgs48', 'cities': 'gy9emg9m', 'persons': 'i75hg48h'}
# some_gemma_runs = {'events': 'uhzc6p3n', 'landmarks': '7674oy2c', 'artworks': 'zzzexkck', 'books': '44pyh719', 'inventions': 'c3zcv8kz', 'movies': 'k6x93v9v', 'cities': '7ejgeyhp', 'persons': 'o03lkeul'}
splits = ['en', 'ru', 'zh', 'de']
some_mistral_runs = {'de': 'ay8eojx5', 'zh': 'tsu5pyek', 'ru': 'k5hge2wp', 'en': 'sboe48ib'}
some_gemma_runs = {'de': 'ay8eojx5', 'zh': 'tsu5pyek', 'ru': 'k5hge2wp', 'en': 'sboe48ib'}

for split in splits:
    print(split, end=' ')
    for runs in [some_mistral_runs, some_gemma_runs]:
        claims = get_base_accuracy(runs[split])
        print('&', claims, end=' ')
    print()

en & 94.9\% & 94.9\% 
ru & 66.7\% & 66.7\% 
zh & 77.8\% & 77.8\% 
de & 85.1\% & 85.1\% 


In [50]:
from collections import defaultdict
from datetime import datetime
from tqdm import tqdm

def _parse_tm_from_log(l):
    timestamp_str = l.split(']')[0].strip('[')
    dt = datetime.strptime(timestamp_str, "%Y-%m-%d %H:%M:%S,%f")
    return dt.timestamp()

def get_stat_calc_times(run_ids):
    total_secs = defaultdict(list)
    for run_id in tqdm(run_ids, total=len(run_ids)):
        lines = get_logs(run_id).split('\n')
        run_total_secs = defaultdict(float)
        for i, l in enumerate(lines):
            if 'Done calculating ' in l and ' sec' in l:
                fn = l.split('Done calculating ', 1)[-1].split(' in ')[0]
                if fn.startswith('<') and fn.endswith('>'):
                    fn = fn.split(' ')[0].split('.')[-1]
                secs = float(l.split(' sec')[0].split(' in ')[-1])
                run_total_secs[fn] += secs
            elif 'Done generating in' in l:
                secs = float(l.split(' sec')[0].split(' in ')[-1])
                run_total_secs['Inference'] += secs
        assert 'Inference' in run_total_secs
        for key, val in run_total_secs.items():
            total_secs[key].append(val)
    return total_secs

def get_times(run_ids, model_nm):
    total_secs = get_stat_calc_times(run_ids.values())
    stat_calc_to_methods = {
        'ApplyUQHeadCalculator': [model_nm],
        'CalculatorApplyUQHead': [model_nm],
        'InferLuqCalculator': [model_nm],
        'CalculatorInferLuh': [model_nm],
        'Inference': ['CCP', 'Max Token Entropy', 'MCP', 'Perplexity'],
        'ClaimsExtractor': [],
        'GreedyAlternativesNLICalculator': ['CCP'],
        'GreedyAlternativesFactPrefNLICalculator': ['CCP'],
        'EntropyCalculator': ['Max Token Entropy'],
        'RandomBaselineClaim': ['Random'],
        'MaximumClaimProbability': ['MCP'],
        'CCP_claim_no_context': ['CCP'],
        'CCP_claim_fact_pref': ['CCP'],
        'MaxTokenEntropyClaim': ['Max Token Entropy'],
        'PerplexityClaim': ['Perplexity'],
        'LuqClaimEstimatorDummy_claim': [model_nm],
        'LuqClaimEstimator_mean_reverse=False': [model_nm],
        'LuqClaimEstimator_mean_logits_reverse=False': [],
        'LuqClaimEstimator_prod_geom_reverse=False': [],
        'OpenAIFactCheck': [],
    }
    runtime = defaultdict(list)
    for key, vals in total_secs.items():
        if key not in stat_calc_to_methods:
            raise Exception(f'Calculated unknown key: {key}')
        for method in stat_calc_to_methods[key]:
            runtime[method] += vals
    return runtime

all_times = [
    get_times( # mistral attn
        {'events': 'peuix8vc', 'landmarks': 'xbvaicss', 'artworks': 'ifx5x00b', 'books': 'xvfzpaqy', 'inventions': 'u2e55em8', 'movies': 'aj9nmvs3', 'cities': '8yhj7nmk', 'persons': '15wrngsa'},
        'Uhead attn',
    ),
    get_times( # mistral factoscope
        {'events': '0k34rsbw', 'landmarks': 'dpiotb3y', 'artworks': '40uteysh', 'books': 'a4nbf6lo', 'inventions': 'dxk2dp92', 'movies': 'cy7rp5jr', 'cities': 'yko0xc47', 'persons': '278wxs1l'},
        'Factoscope',
    ),
    get_times( # mistral lblens
        {'events': 'k29j6kp2', 'landmarks': '70gjmmdb', 'artworks': '1912ptmt', 'books': 'n8t77wzu', 'inventions': 'cdr4n8na', 'movies': '5ijabeer', 'cities': 'un2t458h', 'persons': 'pksvls7j'},
        'Lookback lens',
    ),
    get_times( # gemma saplma, partial
        {'landmarks': '8wpt7qcv', 'artworks': '650nr5rv', 'books': 'xzpo9uju', 'inventions': 'wqyu7thj', 'movies': '47sbspxz', 'cities': 'cy1m1v6t', 'persons': '5m231irc'},
        'SAPLMA',
    )
]

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:12<00:00,  1.84s/it]


In [52]:
import pandas as pd
from collections import defaultdict

comp_eff_table = defaultdict(list)
for runtimes in all_times:
    reference = 'MCP'
    for method, tm in runtimes.items():
#         comp_eff_table[method].append(tm / runtimes[reference] - 1)
        if method not in comp_eff_table.keys():
            comp_eff_table[method] = str(round((sum(tm) / sum(runtimes[reference]) - 1) * 100, 1))
df = pd.DataFrame({'Computational overhead compared to MCP': comp_eff_table})
df = df.reindex(index=[
    'MCP', 'Perplexity', 'Max Token Entropy', 'CCP',
    'SAPLMA', 'Factoscope', 'Lookback lens', 'Uhead attn',
])
print(df.to_latex())
df

\begin{tabular}{ll}
\toprule
 & Computational overhead compared to MCP \\
\midrule
MCP & 0.0 \\
Perplexity & 0.0 \\
Max Token Entropy & 0.2 \\
CCP & 95.3 \\
SAPLMA & 4.7 \\
Factoscope & 6.1 \\
Lookback lens & 5.5 \\
Uhead attn & 15.1 \\
\bottomrule
\end{tabular}



,Computational overhead compared to MCP
MCP,0.0
Perplexity,0.0
Max Token Entropy,0.2
CCP,95.3
SAPLMA,4.7
Factoscope,6.1
Lookback lens,5.5
Uhead attn,15.1


In [54]:
all_times = [
    get_times( # gemma attn
        {'events': 'vdw8no5q', 'landmarks': 'sjls7cbj', 'artworks': 'kn08tkvn', 'books': 'b0m14m8s', 'inventions': 'o50q8ed3', 'movies': '6lbmxy35', 'cities': '3s91bo6y', 'persons': 'o8lnqlk0'},
        'Uhead attn',
    ),
    get_times( # mistral factoscope
        {'events': '0k34rsbw', 'landmarks': 'dpiotb3y', 'artworks': '40uteysh', 'books': 'a4nbf6lo', 'inventions': 'dxk2dp92', 'movies': 'cy7rp5jr', 'cities': 'yko0xc47', 'persons': '278wxs1l'},
        'Factoscope',
    ),
    get_times( # gemma lblens
        {'events': '9hvgpu50', 'landmarks': '625qpm99', 'artworks': 'r3dua7ef', 'books': 'cqaaeu9r', 'inventions': 'zi4wlcri', 'movies': 'q9ckoa2w', 'cities': 'xdhqfwr3', 'persons': 'r5dk6tfj'},
        'Lookback lens',
    ),
    get_times( # gemma saplma
        {'events': 'emyfvbbc', 'landmarks': '8wpt7qcv', 'artworks': '650nr5rv', 'books': 'xzpo9uju', 'inventions': 'wqyu7thj', 'movies': '47sbspxz', 'cities': 'cy1m1v6t', 'persons': '5m231irc'},
        'SAPLMA',
    )
]

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 8/8 [00:14<00:00,  1.82s/it]


In [57]:
import pandas as pd
from collections import defaultdict

comp_eff_table = defaultdict(list)
for runtimes in all_times:
    reference = 'MCP'
    for method, tm in runtimes.items():
#         comp_eff_table[method].append(tm / runtimes[reference] - 1)
        if method not in comp_eff_table.keys():
            comp_eff_table[method] = str(round((sum(tm) / sum(runtimes[reference]) - 1) * 100, 1))
df = pd.DataFrame({'Computational overhead compared to MCP': comp_eff_table})
df = df.reindex(index=[
    'MCP', 'Perplexity', 'Max Token Entropy', 'CCP',
    'SAPLMA', 'Factoscope', 'Lookback lens', 'Uhead attn',
])
print(df.to_latex())
df

\begin{tabular}{ll}
\toprule
 & Computational overhead compared to MCP \\
\midrule
MCP & 0.0 \\
Perplexity & 0.0 \\
Max Token Entropy & 0.9 \\
CCP & 0.0 \\
SAPLMA & 4.7 \\
Factoscope & 6.1 \\
Lookback lens & 9.3 \\
Uhead attn & 18.7 \\
\bottomrule
\end{tabular}



,Computational overhead compared to MCP
MCP,0.0
Perplexity,0.0
Max Token Entropy,0.9
CCP,0.0
SAPLMA,4.7
Factoscope,6.1
Lookback lens,9.3
Uhead attn,18.7
